# Mirage19 Data Exploration

In [ ]:
#	#########
#	LIBRARIES

# Loading the dataset
import pickle

# Data manipulation and analysis
import numpy as np
import pandas as pd
from collections import Counter
from typing import Iterable

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Data preprocessing
# La classe MinMaxScaler serve per scalare le caratteristiche in un intervallo specifico, tipicamente tra 0 e 1.
# Questo è utile per normalizzare i dati prima di applicare algoritmi di machine learning,
# poiché molti algoritmi funzionano meglio quando le caratteristiche hanno scale simili.
from sklearn.preprocessing import MinMaxScaler

In [ ]:
#	#########
#	CONSTANTS

DATA_PATH = "../dataset/mirage/2024/act/mirage2024_act_LOPEZ_lopez_lopez_36P_4F_APP_xST_PAD_cf1a9527.pickle"
""" Path to the dataset file. """

DATASET_PACKETS = 36
""" Number of packets available at most in each flow. """

PADDING_VALUE = -1
""" This value is used in the dataset to denote padding packets. """

N_PACKETS = 10
""" Number of packets to consider in each flow. """

N_FEATURES = 4
""" Number of features for each packet. """

FEATURES_LIST = ['DIR', 'PL', 'TCPWIN', 'IAT']
""" List of names of available features. """


In [ ]:
#	#########
#	FUNCTIONS

def count_packets(flow : Iterable) -> int:
	"""Calcola il numero di pacchetti validi in un singolo flusso.

	Args:
		flow (iterable): sequenza di pacchetti del flusso,
		dove ogni pacchetto è rappresentato da un array di features.
		Le features sono: [DIR (0), PL (1), TCPWIN (2), IAT (3)].

	Returns:
		int: restituisce l'indice del primo pacchetto di padding, identificato dal
		valore -1 nella prima feature del pacchetto.
		Se non viene trovato alcun pacchetto di padding, restituisce la
		lunghezza totale del flusso.
	"""

	# Cerca il primo pacchetto di padding nel flusso.
	return next((

		# Identifica tutti gli indici dei pacchetti nel flusso.
		i for i, packet in enumerate(flow)

		# Il padding è identificato dal valore -1 nella prima feature (DIR).
		if packet[0] == -1),

		# Se non viene trovato alcun padding, restituisce la lunghezza del flusso.
		len(flow)
	)

	# end

def log1pPreprocessing(X : np.ndarray, num_packets : int = N_PACKETS):
	""" Applica il preprocessing ai flussi di traffico,
	con masking del padding e trasformazione logaritmica.
	
	Args:
		X (numpy.ndarray): tensor con shape (N, X, 4),
		dove N è il numero di flussi, X è il numero massimo di pacchetti per flusso
		(può essere 36 o 100 a seconda del dataset) e 4 è il numero di features per pacchetto.
		Le features sono: [DIR (0), PL (1), TCPWIN (2), IAT (3)].
		
		num_packets (int, optional): numero di pacchetti da considerare per flusso.
		Default è N_PACKETS.

	Returns:
		numpy.ndarray: Tensor preprocessato con shape (N, N_PACKETS, 4), in
		formato float32. I pacchetti di padding vengono azzerati e le feature
		PL, TCPWIN e IAT sono trasformate con "np.log1p".
	"""

	# Copia l'input per evitare modifiche in-place.
	X_proc = X.copy().astype(np.float32)

	# Identifica i pacchetti di padding (dove la feature PL è -1).
	is_padding = (X_proc[:, :, 1] == -1)

	# Azzera tutte le feature dei pacchetti di padding.
	mask = np.repeat(is_padding[:, :, np.newaxis], 4, axis=2)
	X_proc[mask] = 0

	# Applica la trasformazione logaritmica (Log1p)
	# alle feature PL (1), TCPWIN (2) e IAT (3).
	X_proc[:, :, 1:] = np.log1p(np.maximum(X_proc[:, :, 1:], 0))

	# Tronca la sequenza ai primi num_packets e ritorna il risultato.
	return X_proc[:, :num_packets, :]

	# end

def minMaxPreprocessing(X : np.ndarray, num_packets : int = N_PACKETS):
	"""Applica una normalizzazione min-max alle feature dei pacchetti. 

	Args:
		X (numpy.ndarray): tensor con shape (N, X, 4),
		dove N è il numero di flussi, X è il numero massimo di pacchetti per flusso
		(può essere 36 o 100 a seconda del dataset) e 4 è il numero di features per pacchetto.
		Le features sono: [DIR (0), PL (1), TCPWIN (2), IAT (3)].
		
		num_packets (int, optional): numero di pacchetti da considerare per flusso.
		Default è N_PACKETS.

	Returns:
		numpy.ndarray: tensor preprocessato con shape (N, N_PACKETS, 4),
		in formato float32, dove tutte le feature dei pacchetti sono scalate
		nell'intervallo [0, 1] usando la normalizzazione min-max.
	"""

	# Copia l'input per evitare modifiche in-place.
	X_proc = X.copy().astype(np.float32)

	# Inizializza lo scaler MinMaxScaler.
	scaler = MinMaxScaler(feature_range=(0, 1))

	# Adatta i dati in una forma che MinMaxScaler può trattare correttamente.
	# MinMaxScaler.fit() non lavora direttamente su tensori 3D, ma vuole dati
	# organizzati come campioni per righe e feature per colonne.
	# Il valore -1 serve a inferire il numero di righe necessario in base al
	# numero totale di elementi e al numero di feature fissato.
	scaler.fit(np.reshape(X_proc, [-1, N_FEATURES]))

	# Applica la trasformazione min-max a tutte le feature dei pacchetti.
	X_proc = scaler.transform(np.reshape(X_proc, [-1, N_FEATURES]))

	# Ripristina la forma originale del tensor.
	X_proc = np.reshape(X_proc, [-1, X.shape[1], N_FEATURES])

	# Tronca la sequenza ai primi num_packets e ritorna il risultato.
	return X_proc[:, :num_packets, :]

	# end

def inspect_feature_preprocessing(
	sample_idx: int,
	feature_idx: int,
	X_pp: dict[np.ndarray],
	y_raw: np.ndarray,
	n_packets: int = N_PACKETS,
	show_stats: bool = True,
	y_limits: tuple | None = None
):
	"""Visualizza e confronta una feature prima e dopo
	l'applicazione delle funzioni di preprocessing.

	La funzione, dato un campione e una feature, mostra l'andamento della feature
	nella versione originale, e nelle versioni preprocessate passate tramite
    dizionario (es. log1p, min-max, ...).
	Facoltativamente stampa anche statistiche di base sui valori validi.

	Args:
		sample_idx (int): Indice del campione da analizzare.
		feature_idx (int): Indice della feature da visualizzare.
		X_pp (dict[numpy.ndarray]): Dizionario contenente il dataset originale e quelli
		preprocessati, nel quale la chiave è il nome della funzione di preprocessing,
		il valore è il relativo array preprocessato.
		y_raw (numpy.ndarray): Vettore delle etichette.
		n_packets (int, optional): Numero massimo di pacchetti da visualizzare.
		show_stats (bool, optional): Se True, stampa min, max e delta sui valori validi.
		y_limits (tuple | None, optional): Limiti dell'asse y per i tre subplot.
	"""

	#   ####################################################################    #
    #   INIZIALIZZAZIONE

	feature_name = FEATURES_LIST[feature_idx]
	colors = ['orange', 'green', 'red', 'purple', 'brown', 'blue']

	print("Random sample index:", sample_idx, "App label:", y_raw[sample_idx])

	#   ####################################################################    #
	#	ANALISI DELLE FEATURE SUI DATASET PREPROCESSATI disponibili

	# Crea un subplot per il dataset originale e uno per ogni preprocessing.
	n_plots = len(X_pp)
	_, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 5))

	for i, (pp_name, X_curr) in enumerate(X_pp.items()):
		
		limit = n_packets if pp_name == "Original" else -1
		pad_value = PADDING_VALUE if pp_name == "Original" else 0

		# Estrazione.
		pp_values = X_curr[sample_idx, :limit, feature_idx]

		# Filtra i pacchetti validi.
		pp_valid = pp_values[pp_values != pad_value]

		if show_stats and len(pp_valid) > 0:
			
			# Calcola le statistiche.
			pp_min = np.min(pp_valid)
			pp_max = np.max(pp_valid)
			pp_delta = pp_max - pp_min

			# Stampa le statistiche.
			print(
				f"{pp_name} {feature_name}\n" +
				f"min: {pp_min:.4f} " +
				f"max: {pp_max:.4f} " +
				f"delta: {pp_delta:.4f}"
			)
		
		color = colors[(i - 1) % len(colors)]
		
		axes[i].plot(pp_values, marker='o', color=color)
		axes[i].set_title(f"{pp_name} {feature_name}")
		axes[i].set_xlabel("Packet Index")

		if y_limits is not None and pp_name in y_limits:
			axes[i].set_ylim(y_limits[pp_name])
		
		# end for pp_name, X_curr

	plt.tight_layout()
	plt.show()

	# end

## Load Data
Loading the preprocessed pickle file containing biflows and labels.

In [ ]:
print("Loading dataset...")

# Il file pickle contiene due oggetti salvati in sequenza:
# 1. X_raw : i dati numerici dei flussi di traffico, in formato numpy array.
# 2. y_raw : le etichette corrispondenti ai flussi.
# Ogni chiamata a pickle.load() legge il successivo oggetto nel file.
with open(DATA_PATH, "rb") as f:
    X_raw = np.array(pickle.load(f), dtype=np.float32)
    y_raw = np.array(pickle.load(f))

print(f"Data shape: {X_raw.shape}")
print(f"Labels shape: {len(y_raw)}")

# Si definisce un campione d'esempio per tutto il notebook.
example_sample = 7000
print("\nExample Sample (First 5 packets):\n", X_raw[example_sample][:5])
print("Example Label:", y_raw[example_sample])

## Data exploration

In [ ]:
# Inizializzazione delle liste che conterranno, per ogni flusso,
# le statistiche riassuntive calcolate sulla parte non-padding.
X_n_packets = []
X_dir_trimmed = []
X_payloadLen_trimmed = []
X_tcpWin_trimmed = []
X_iat_trimmed = []

# Lettura dei flussi grezzi ed estrazione delle statistiche sui pacchetti validi.
for flow in X_raw:
    
	# Conteggio dei pacchetti.
    num_packets = count_packets(flow)
    X_n_packets.append(num_packets)

	# Estrazione della direzione.
    X_dir_trimmed.append(flow[:num_packets, 0])
    
	# Calcolo della somma della lunghezza del payload.
    X_payloadLen_trimmed.append(flow[:num_packets, 1].sum())
    
	# Calcolo della lunghezza media della finestra TCP.
    X_tcpWin_trimmed.append(flow[:num_packets, 2].mean())
    
	# Calcolo del tempo medio dell'inter-arrival time (IAT).
    X_iat_trimmed.append(flow[:num_packets, 3].mean())
    
	# end for flow

# Costruisce un dataframe con la label e le feature riassuntive estratte da ciascun flusso.
df = pd.DataFrame({
    'y': y_raw,
    'x': np.array(X_n_packets),
    'dir': X_dir_trimmed,
    'payloadLen': X_payloadLen_trimmed,
    'tcpWin': X_tcpWin_trimmed,
    'iat': X_iat_trimmed,
})

In [ ]:
# Stampa il campione di esempio
print(df.iloc[example_sample], "\n")
# print(df[df['y'] == "Diretta"], "\n")

print("Mean packets per flow: ", np.mean(X_n_packets), "\n")
print("Standard deviation packets per flow: ", np.std(X_n_packets))

In [ ]:
# Calcola tutte le statistiche riassuntive per ciascuna classe (y) sul numero di pacchetti (x).
# Si utilizza il metodo reset_index() per trasformare
# il risultato del groupby() in un DataFrame standard, così da poter
# rinominare le colonne e ordinarle facilmente per numero di pacchetti
# utilizzando il metodo sort_values().
df_summary = df.groupby('y')['x'].agg([
	'count',
	'mean', 'std', 'median',
    'min', 'max',
	pd.Series.skew, pd.Series.kurt
]).sort_values('count', ascending=False).reset_index()

print(df_summary)

### Class Distribution

**Questo grafico mostra come il dataset è distribuito tra le diverse classi di applicazioni.**

Aiuta a verificare se il dataset è bilanciato o se alcune app sono significativamente più rappresentate di altre, aspetto importante per interpretare le prestazioni del modello e individuare potenziali squilibri tra le classi.

In [ ]:
fig, ax = plt.subplots(figsize=(20, 7))

# Disegna un grafico a barre con le etichette delle app
# sull'asse x e il numero di campioni sull'asse y.
sns.barplot(
    data=df_summary,
    x='y', y='count',
    palette='viridis', hue='y',
    ax=ax
)

# Aggiunge il numero esatto di campioni sopra ogni barra.
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', padding=3)

ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
ax.set_xlabel("App")
ax.set_ylabel("Number di campioni per classe")
ax.set_title("Class Distribution", fontsize=16)

plt.tight_layout()
plt.show()

## Numero medio di pacchetti e Deviazione standard

**Questo grafico mostra, per ciascuna applicazione, il numero medio di pacchetti validi osservato nei flussi appartenenti a quella classe, insieme alla corrispondente deviazione standard, cioè quanto i campioni di una stessa classe tendono a variare rispetto alla media.**

L'obiettivo è evidenziare eventuali differenze tra le applicazioni in termini di lunghezza media dei flussi, così da individuare pattern di traffico caratteristici che potrebbero risultare utili nelle fasi successive di analisi e classificazione.

La deviazione standard è una misura di dispersione, utile per capire quanto una feature sia stabile o irregolare all'interno di ciascuna categoria: valori bassi indicano che i flussi della stessa classe sono abbastanza omogenei, mentre valori alti segnalano una maggiore variabilità tra i campioni.

Nel contesto dell'analisi del traffico di rete, la deviazione standard può aiutare a identificare applicazioni con comportamento più consistente e applicazioni con pattern più variabili. Questa differenza può essere utile sia per interpretare meglio il dataset sia per valutare se alcune classi siano più difficili da distinguere rispetto ad altre durante la classificazione.

---

*ESEMPIO*

*Per i flussi di "Diretta", una media di 14 pacchetti con deviazione standard 17 indica che la distribuzione è molto dispersa rispetto al valore medio. In pratica, il numero di pacchetti non è concentrato attorno a 14, ma varia parecchio da flusso a flusso: alcuni flussi sono molto brevi, altri molto più lunghi.*

*Dato che la STD è più grande della media, il dato va interpretato con cautela: non significa che i flussi “oscillano di 17 attorno a 14” in modo simmetrico, ma che esiste una variabilità forte e probabilmente una distribuzione non molto concentrata, magari influenzata da sottogruppi o outlier. In un contesto di classificazione, una classe così eterogenea può essere più difficile da modellare perché i campioni non condividono una struttura molto stabile.*

In [ ]:
df_plot = df_summary.sort_values('mean', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(30, 8))

sns.barplot(
    data=df_plot,
    x='y',
    y='mean',
    palette='viridis',
    errorbar=None,
    ax=ax
)

ax.errorbar(
    x=range(len(df_plot)),
    y=df_plot['mean'],
    yerr=df_plot['std'],
    fmt='none',
    c='black',
    capsize=4
)

offset = (df_plot['std'].max() * 0.03)

for i, row in df_plot.iterrows():
    ax.annotate(
        f"{row['mean']:.0f} ± {row['std']:.0f}",
        (i, row['mean'] + row['std'] + offset),
        ha='center',
        va='bottom',
        fontsize=9
    )

ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
ax.set_xlabel("App")
ax.set_ylabel("Numero medio di pacchetti")
ax.set_title("Mean ± Standard Deviation", fontsize=16)

plt.tight_layout()
plt.show()

In [ ]:
classe = "chat"
x = df.loc[df['y'] == classe, 'x']
mean, std = df_summary[df_summary['y'] == classe][['mean', 'std']].iloc[0]

plt.figure(figsize=(6, 6))
ax = sns.histplot(x, bins=DATASET_PACKETS, kde=True, color='steelblue', edgecolor='white')

plt.axvline(mean, color='red', linestyle='--', linewidth=2, label=f"Mean = {mean:.2f}")
plt.axvline(mean - std, color='green', linestyle=':', linewidth=2, label=f"Mean - STD = {mean-std:.2f}")
plt.axvline(mean + std, color='green', linestyle=':', linewidth=2, label=f"Mean + STD = {mean+std:.2f}")

plt.xticks([i for i in range(0, DATASET_PACKETS, DATASET_PACKETS//10)] + [DATASET_PACKETS])
plt.title(f"Distribution of packet counts - {classe}")
plt.xlabel("Number of packets")
plt.ylabel("Number of flows")

plt.legend()
plt.tight_layout()
plt.show()

## Preprocessing

In [ ]:
# Si applicano le funzioni di preprocessing al dataset completo.
X_pp = {
    "Original" : X_raw,
    "log1p": log1pPreprocessing(X_raw),
    "min-max": minMaxPreprocessing(X_raw),
}

# Si scelgono due campioni per confrontare l'input originale e le versioni preprocessate
samples_to_inspect = [52180, 44261]

for sample_idx in samples_to_inspect:
    inspect_feature_preprocessing(
        sample_idx=sample_idx,
        feature_idx=1,
        X_pp=X_pp, y_raw=y_raw,
        show_stats=True,
        y_limits={
            "Original": (-10, np.max(X_raw[:, :, 1]) * 1.1),
            "log1p": (-0.1, np.max(X_pp["log1p"][:, :, 1]) * 1.1),
            "min-max": (-0.1, 1.1),
        }
    )

for sample_idx in samples_to_inspect:
    inspect_feature_preprocessing(
        sample_idx=sample_idx,
        feature_idx=0,
        X_pp=X_pp, y_raw=y_raw,
        show_stats=False,
        y_limits={
            "Original": (-1.1, 1.1),
            "log1p": (-1.1, 1.1),
            "min-max": (-1.1, 1.1),
        }
    )